<a href="https://colab.research.google.com/github/HereLiesAz/PaperPlanes/blob/main/PaperPlanes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "rembg[gpu]" torch diffusers transformers accelerate opencv-python-headless pillow numpy requests
import kagglehub
path = kagglehub.model_download('vaishaknair456/u2-net-portrait-background-remover/tfLite/40/1')



  0%|          | 0.00/168M [00:00<?, ?B/s]
  1%|          | 1.00M/168M [00:01<03:45, 775kB/s]
  1%|          | 2.00M/168M [00:01<01:55, 1.51MB/s]
  2%|▏         | 3.00M/168M [00:01<01:12, 2.38MB/s]
  3%|▎         | 5.00M/168M [00:01<00:38, 4.40MB/s]
  4%|▍         | 7.00M/168M [00:01<00:25, 6.73MB/s]
  5%|▌         | 9.00M/168M [00:02<00:19, 8.72MB/s]
  7%|▋         | 11.0M/168M [00:02<00:15, 11.0MB/s]
  8%|▊         | 13.0M/168M [00:02<00:13, 12.1MB/s]
  9%|▉         | 15.0M/168M [00:02<00:11, 13.8MB/s]
 10%|█         | 17.0M/168M [00:02<00:10, 15.0MB/s]
 11%|█▏        | 19.0M/168M [00:02<00:09, 16.1MB/s]
 13%|█▎        | 21.0M/168M [00:02<00:09, 16.1MB/s]
 14%|█▎        | 23.0M/168M [00:02<00:09, 16.7MB/s]
 15%|█▍        | 25.0M/168M [00:03<00:08, 17.0MB/s]
 16%|█▌        | 27.0M/168M [00:03<00:08, 16.8MB/s]
 17%|█▋        | 29.0M/168M [00:03<00:08, 17.6MB/s]
 18%|█▊        | 31.0M/168M [00:03<00:08, 17.1MB/s]
 20%|█▉        | 33.0M/168M [00:03<00:07, 17.9MB/s]
 21%|██        | 35.0

In [3]:
!git clone https://github.com/HereLiesAz/PaperPlanes.git
import os

repo_path = 'PaperPlanes'
if os.path.exists(repo_path):
    print("Repository structure:")
    for root, dirs, files in os.walk(repo_path):
        level = root.replace(repo_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{sub_indent}{f}")

Cloning into 'PaperPlanes'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 306 (delta 43), reused 32 (delta 32), pack-reused 250 (from 2)
Receiving objects: 100% (306/306), 17.09 MiB | 16.43 MiB/s, done.
Resolving deltas: 100% (151/151), done.
Repository structure:
PaperPlanes/
    requirements.txt
    api.py
    sw.js
    app.js
    PaperPlanes.ipynb
    vivisect_art.py
    index.html
    slaughterhouse.py
    pipeline.py
    .gitignore
    manifest.json
    README.md
    wrangler.jsonc
    .git/
        packed-refs
        description
        config
        index
        HEAD
        refs/
            remotes/
                origin/
                    HEAD
            heads/
                main
            tags/
        logs/
            HEAD
            refs/
                remotes/
                    origin/
                        HEAD
                heads/
               

In [ ]:
import os
import io
import zipfile
import cv2
import torch
import numpy as np
import hashlib
import requests
import re
from PIL import Image
from rembg import remove
from diffusers import StableDiffusionImg2ImgPipeline
from transformers import pipeline as hf_pipeline
from google.colab import drive
from google.colab import files
from IPython.display import display

print("Authenticating with Google Drive...")
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/PaperPlanes_Output"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Output directory secured at: {DRIVE_DIR}")

def get_file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

def display_thumbnail(title, img, max_size=400):
    """Helper function to display process images cleanly in the Colab notebook."""
    print(f"  -> Displaying: {title}")
    display_img = img.copy()
    display_img.thumbnail((max_size, max_size))
    display(display_img)

def process_image(image_path, layers=6):
    base_name = os.path.basename(image_path).split('.')[0]
    zip_path = os.path.join(DRIVE_DIR, f"{base_name}_layers.zip")

    if os.path.exists(zip_path):
        print(f"Artifact already exists: {zip_path}. Skipping.")
        return

    print(f"\n=== Processing: {image_path} ===")

    # Phase 1
    print("Phase 1: Removing background...")
    orig_pil = Image.open(image_path).convert("RGB")
    nobg_pil = remove(orig_pil)
    subject_mask = np.array(nobg_pil)[:, :, 3] > 0
    orig_cv = cv2.cvtColor(np.array(orig_pil), cv2.COLOR_RGB2BGR)
    display_thumbnail("Isolated Subject", nobg_pil)

    # Phase 2
    print("Phase 2: Generating structural reference...")
    sd_pipe = StableDiffusionImg2ImgPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
    sd_pipe.safety_checker = None

    prompt = "Raw, hyper-realistic photograph, incredibly detailed, 8k resolution, cinematic lighting, physical reality, sharp focus, real life"
    negative_prompt = "painting, illustration, drawing, art, canvas, brushstrokes, graffiti, wall, background, sketch, 2d, flat, texture, graphic, stylized"

    gen_pil = sd_pipe(prompt=prompt, negative_prompt=negative_prompt, image=orig_pil, strength=0.85, guidance_scale=12.0).images[0]
    gen_cv = cv2.cvtColor(np.array(gen_pil), cv2.COLOR_RGB2BGR)
    del sd_pipe
    torch.cuda.empty_cache()
    display_thumbnail("Generated Image", gen_pil)

    # Phase 3
    print("Phase 3: Aligning generated image to source...")
    gen_cv = cv2.resize(gen_cv, (orig_cv.shape[1], orig_cv.shape[0]))
    gray_src = cv2.cvtColor(orig_cv, cv2.COLOR_BGR2GRAY)
    gray_tgt = cv2.cvtColor(gen_cv, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(MAX_FEATURES=5000)
    kp_src, des_src = orb.detectAndCompute(gray_src, None)
    kp_tgt, des_tgt = orb.detectAndCompute(gray_tgt, None)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    matches = bf.knnMatch(des_src, des_tgt, k=2)
    good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]

    if len(good_matches) > 10:
        src_pts = np.float32([kp_src[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        tgt_pts = np.float32([kp_tgt[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        matrix, _ = cv2.findHomography(tgt_pts, src_pts, cv2.RANSAC, 5.0)
        aligned_cv = cv2.warpPerspective(gen_cv, matrix, (orig_cv.shape[1], orig_cv.shape[0])) if matrix is not None else gen_cv
    else:
        aligned_cv = gen_cv

    aligned_pil = Image.fromarray(cv2.cvtColor(aligned_cv, cv2.COLOR_BGR2RGB))
    display_thumbnail("Aligned Image", aligned_pil)

    # Phase 4
    print("Phase 4: Extracting depth map...")
    depth_pipe = hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
    depth_array = np.array(depth_pipe(aligned_pil)["depth"]).astype(np.float32)
    del depth_pipe
    torch.cuda.empty_cache()

    # Normalize depth array purely for visual feedback in Colab
    depth_vis = (depth_array - depth_array.min()) / (depth_array.max() - depth_array.min() + 1e-8)
    depth_vis = (depth_vis * 255).astype(np.uint8)
    display_thumbnail("Depth Map", Image.fromarray(depth_vis))

    # Phase 5
    print("Phase 5: Segmenting layers...")
    subject_depth = depth_array[subject_mask]
    min_d, max_d = subject_depth.min(), subject_depth.max()
    normalized_depth = np.zeros_like(depth_array)
    if min_d != max_d:
        normalized_depth[subject_mask] = np.interp(depth_array[subject_mask], (min_d, max_d), (0, 255))

    bins = np.linspace(0, 255.1, layers + 1)
    orig_array = np.array(orig_pil)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for i in range(layers):
            layer_mask = (normalized_depth >= bins[i]) & (normalized_depth < bins[i+1]) & subject_mask
            if np.any(layer_mask):
                layer_rgba = np.zeros((orig_array.shape[0], orig_array.shape[1], 4), dtype=np.uint8)
                layer_rgba[..., :3] = orig_array
                layer_rgba[..., 3] = cv2.GaussianBlur((layer_mask * 255).astype(np.uint8), (5, 5), 0)
                img_byte_arr = io.BytesIO()
                Image.fromarray(layer_rgba).save(img_byte_arr, format='PNG')
                zf.writestr(f"layer_{i:03d}.png", img_byte_arr.getvalue())

    print(f"Segmentation complete. Zip file permanently saved to Drive: {zip_path}")


def download_album_images(shared_url):
    print(f"Connecting to Google Photos URL: {shared_url}")
    response = requests.get(shared_url)

    matches = re.findall(r'(https:\/\/lh3\.googleusercontent\.com\/[a-zA-Z0-9\-_]+)', response.text)
    unique_urls = list(set(matches))

    image_urls = [url for url in unique_urls if len(url) > 60]

    if not image_urls:
        print("No images found. Ensure you provided a valid 'Create link' share URL.")
        return []

    os.makedirs("source_images", exist_ok=True)
    downloaded_files = []

    for i, url in enumerate(image_urls):
        try:
            img_data = requests.get(url + "=d").content
            filename = f"source_images/source_{i:03d}.jpg"
            with open(filename, "wb") as f:
                f.write(img_data)
            downloaded_files.append(filename)
            print(f"Downloaded: {filename}")
        except Exception as e:
            print(f"Failed to download {url}: {e}")

    return downloaded_files

# ---------------------------------------------------------
# EXECUTION ROUTINE
# ---------------------------------------------------------
ALBUM_URL = "" # Paste your Google Photos shared album URL here. Leave blank for manual upload.

processed_hashes = set()
valid_extensions = {'.png', '.jpg', '.jpeg', '.webp'}

if ALBUM_URL:
    print("Album URL detected. Initiating download...")
    target_files = download_album_images(ALBUM_URL)
    for target in target_files:
        file_hash = get_file_hash(target)
        if file_hash in processed_hashes:
            print(f"Skipping duplicate file from album.")
            continue
        processed_hashes.add(file_hash)
        process_image(target, layers=6)
else:
    print("No Album URL provided. Initiating manual Colab upload widget...")
    uploaded = files.upload()
    for filename in uploaded.keys():
        ext = os.path.splitext(filename)[1].lower()
        if ext not in valid_extensions:
            print(f"Skipping non-image file: {filename}")
            continue

        file_hash = get_file_hash(filename)
        if file_hash in processed_hashes:
            print(f"Skipping duplicate file: {filename}")
            continue

        processed_hashes.add(file_hash)
        process_image(filename, layers=6)

Authenticating with Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory secured at: /content/drive/MyDrive/PaperPlanes_Output
No Album URL provided. Initiating manual Colab upload widget...
